# Credit Card Fraud Detection — EDA refatorado

## Objetivo

Explorar o dataset antes da modelagem, mantendo o notebook focado em **execução + interpretação**.

As rotinas genéricas de EDA foram extraídas para `src/eda_utils.py`.

```text
entendimento inicial
↓
split treino / validação / holdout
↓
EDA apenas no treino
↓
target
↓
Amount e Time
↓
V1–V28
↓
correlação
↓
outliers
↓
conclusões
```

> `X_val / y_val` e `X_holdout / y_holdout` não participam da EDA orientada ao target.


## 1. Imports e configurações

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split


from src.eda_utils import (
    bar_plot,
    boxplot_by_class,
    correlation_plot,
    get_outlier_summary,
    histogram_plot,
    kde_by_class,
    plot_outlier_summary,
    summary_statistics,
)


TARGET = "Class"
RANDOM_STATE = 42

## 2. Carregamento e entendimento inicial

In [ ]:
df = pd.read_csv("data/raw/creditcard.csv")

print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
print("Valores ausentes:", df.isna().sum().sum())
print("Linhas duplicadas:", df.duplicated().sum())

display(df.describe().T)

### Observação

Nesta etapa apenas entendemos a estrutura da base. Duplicatas e valores extremos não são removidos automaticamente, porque em detecção de fraude observações raras podem carregar sinal relevante.


## 3. Split antes da EDA orientada ao target

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_dev,
)

train = X_train.copy()
train[TARGET] = y_train

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)
print("Holdout:", X_holdout.shape)

In [ ]:
print("Treino (%)")
display(y_train.value_counts(normalize=True).mul(100))

print("Validação (%)")
display(y_val.value_counts(normalize=True).mul(100))

print("Holdout (%)")
display(y_holdout.value_counts(normalize=True).mul(100))

## 4. Target

**Pergunta:** quão desbalanceado é o problema?


In [ ]:
bar_plot(train, TARGET)
plt.show()

**Interpretação:** o target é extremamente desbalanceado. Na modelagem, `accuracy` não deve ser usada isoladamente.


## 5. `Amount` e `Time`

Começamos pelas duas variáveis que não são componentes PCA.


In [ ]:
summary_statistics(
    train,
    ["Amount", "Time"],
)

### 5.1 `Amount`

`Amount` apresenta forte assimetria. Primeiro observamos a distribuição original; depois criamos `Amount_log` apenas para facilitar a visualização por classe.


In [ ]:
histogram_plot(train, "Amount")
plt.show()

In [ ]:
train["Amount_log"] = np.log1p(train["Amount"])

boxplot_by_class(
    train,
    "Amount_log",
    TARGET,
)

plt.show()

**Leitura:** existe sobreposição considerável entre as classes. `Amount` isoladamente não é suficiente para separar fraude de transação legítima.


### 5.2 `Time`

In [ ]:
kde_by_class(
    train,
    "Time",
    TARGET,
)

plt.show()

**Leitura:** `Time` pode apresentar regiões de concentração diferentes, mas também existe forte sobreposição entre as classes.


## 6. Componentes `V1` a `V28`

**Pergunta:** quais componentes apresentam maior diferença entre fraude e transação legítima?

A diferença absoluta entre médias é apenas uma **heurística exploratória**, não feature importance.


In [ ]:
v_features = [f"V{i}" for i in range(1, 29)]

mean_by_class = train.groupby(TARGET)[v_features].mean().T

mean_by_class["difference"] = mean_by_class[1] - mean_by_class[0]
mean_by_class["abs_difference"] = mean_by_class["difference"].abs()

ranked_features = mean_by_class.sort_values(
    "abs_difference",
    ascending=False,
)

top10 = ranked_features.head(10)
top5 = ranked_features.head(5).index.to_list()

display(top10)
print("Top 5:", top5)

In [ ]:
plot_data = top10["abs_difference"].sort_values()

plt.figure(figsize=(9, 6))
plt.barh(plot_data.index, plot_data.values)
plt.title("Top 10 features por diferença de média entre classes")
plt.xlabel("Diferença absoluta de média")
plt.show()

### 6.1 Top 5 por classe

In [ ]:
for feature in top5:

    boxplot_by_class(
        train,
        feature,
        TARGET,
    )

    plt.show()

**Leitura:** algumas `V` apresentam diferenças maiores entre as classes, mas nenhuma separa perfeitamente fraude de não fraude. Isso reforça a necessidade de modelos multivariados.


## 7. Correlação com `Class`

In [ ]:
correlation_plot(
    train.drop(columns=["Amount_log"], errors="ignore"),
    TARGET,
    top_n=15,
)

plt.show()

**Importante:** correlação não significa causalidade e não equivale a feature importance.


## 8. Outliers

In [ ]:
outlier_features = v_features + ["Amount", "Time"]

outlier_summary = get_outlier_summary(
    train,
    outlier_features,
)

display(outlier_summary.head(15))

In [ ]:
plot_outlier_summary(
    outlier_summary,
    top_n=15,
)

plt.show()

### Interpretação

Os outliers são identificados, mas não removidos automaticamente. Em detecção de fraude, valores extremos podem representar justamente comportamento anômalo relevante.


## 9. Conclusões da EDA

- O target é extremamente desbalanceado.
- `Amount` é fortemente assimétrico e apresenta sobreposição entre as classes.
- `Time` também não separa as classes isoladamente.
- Alguns componentes `V1–V28` apresentam diferenças maiores entre fraude e não fraude.
- Correlação é usada como evidência exploratória, não como causalidade ou importância preditiva.
- Valores extremos não são removidos automaticamente.
- Nenhuma variável isolada resolve o problema.

A continuação ocorre em `modeling_credit_fraude.ipynb`.
